In [23]:
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = "browser"

panel = pd.read_csv("../data/processed/merged_panel.csv")
panel.head()

,State,Year,Area_1000ha,Prod_1000tons,Yield_t_ha,Kharif_Rain_mm
0,Andhra Pradesh,2000,2694.75,8040.68,2.983832,832.93
1,Andhra Pradesh,2001,2515.37,7823.69,3.110354,509.79
2,Andhra Pradesh,2002,1867.00,5315.00,2.846813,380.95
3,Andhra Pradesh,2003,1957.32,6054.11,3.093061,501.31
4,Andhra Pradesh,2004,2227.69,7392.67,3.318536,490.10


In [24]:
state_list = sorted(panel['State'].unique())
state_list[:10]

['Andhra Pradesh',
 'Assam',
 'Bihar',
 'Chhattisgarh',
 'Gujarat',
 'Haryana',
 'Himachal Pradesh',
 'Jharkhand',
 'Karnataka',
 'Kerala']

In [25]:
def dual_plot(state):
    df = panel[panel.State == state].sort_values("Year")

    fig = go.Figure()

    # Rainfall on left axis
    fig.add_trace(go.Scatter(
        x=df.Year, y=df.Kharif_Rain_mm,
        name="Rainfall (mm)",
        mode="lines+markers",
        yaxis="y1"
    ))

    # Yield on right axis
    fig.add_trace(go.Scatter(
        x=df.Year, y=df.Yield_t_ha,
        name="Yield (t/ha)",
        mode="lines+markers",
        yaxis="y2"
    ))

    fig.update_layout(
        title=f"Rainfall vs Yield — {state} (2000–2019)",
        xaxis=dict(title="Year"),
        yaxis=dict(title="Rain (mm)", side="left"),
        yaxis2=dict(title="Yield (t/ha)", overlaying="y", side="right")
    )
    return fig



In [26]:
def regression_plot(state):
    df = panel[panel.State == state]
    sns.lmplot(data=df, x="Kharif_Rain_mm", y="Yield_t_ha", height=5, aspect=1.3)

In [27]:
import statsmodels.api as sm

def sensitivity(state):
    df = panel[panel.State == state]
    X = sm.add_constant(df.Kharif_Rain_mm)
    model = sm.OLS(df.Yield_t_ha, X).fit()
    return model.params.Kharif_Rain_mm

In [29]:
state = "Bihar"
df = panel[panel.State == state]

X = sm.add_constant(df.Kharif_Rain_mm)
model = sm.OLS(df.Yield_t_ha, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             Yield_t_ha   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.053
Method:                 Least Squares   F-statistic:                   0.04704
Date:                Sun, 18 Jan 2026   Prob (F-statistic):              0.831
Time:                        14:49:00   Log-Likelihood:                -16.284
No. Observations:                  20   AIC:                             36.57
Df Residuals:                      18   BIC:                             38.56
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              1.6460      0.552      2.